# UniXcoder: Big-O from code only vs. code + structural context (3 seeds)
Three experiments, same data, same family-held-out validation split, each repeated with **3 training seeds**:

| Experiment | Input at training and test time | Extra training signal |
|---|---|---|
| **E2 baseline** | source code only | none |
| **E3 side task** | source code only | structural side tasks (loop depth, flags) in the loss |
| **E4 code + structure** | source code **+ structure summary** from the parser, e.g. `[structure] loop_depth=2 dependent_loop=1 ...` | none |

E2 vs E4 answers your question: *does giving the model the structural context as input help it predict Big-O?* In E4 the structure comes from the parser run on the code (never from the Big-O label), so a real pull-request tool can compute it for any changed method.

Results are reported as **mean +- std over the seeds**, so you can tell real differences from noise.

**Time:** about 5 minutes per run, 9 runs = about 45-60 minutes. Use **Save Version -> Save & Run All** if you want it to run in the background. The run loop is resumable: if it stops, rerun Cell 6 and it skips finished runs.

**Before running:** GPU on, Internet on, and the 4 dataset files attached (`train.jsonl`, `validation.jsonl`, `test.jsonl`, `external_test.jsonl`).

| Part | Cells |
|---|---|
| A | 1-3 Setup, load, structural labels, held-out validation split |
| B | 4-5 Tokenize, model and training loop |
| C | 6 Run all experiments and seeds |
| D | 7-10 Summary tables, save, try a before/after pair |

## Cell 1: Setup (must run first)

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

!pip install -q transformers scikit-learn javalang

## Cell 2: Imports and settings

In [2]:
import glob, json, random, time, gc
import numpy as np
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

# ---- settings (edit here only) ----
MODEL_NAME = "microsoft/unixcoder-base"
SAVE_DIR   = "/kaggle/working/my_e4_model"
SEEDS      = [42, 43, 44]     # training seeds (the data split stays fixed)
SPLIT_SEED = 42               # fixed split of train / held-out validation families
EPOCHS     = 5                # the best epoch was epoch 2 before, so 5 is plenty
PATIENCE   = 2
LR         = 2e-5
BATCH      = 16
LAMBDA_AUX = 0.5              # weight of the structural side tasks in E3
VAL_FAMILY_FRACTION = 0.15
MAXLEN     = 512

# name -> (lambda for side tasks, use structure as input)
EXPERIMENTS = {
    "E2 code only":      (0.0,        False),
    "E3 side task":      (LAMBDA_AUX, False),
    "E4 code+structure": (0.0,        True),
}

hits = glob.glob("/kaggle/input/**/train.jsonl", recursive=True)
assert hits, "train.jsonl not found: attach the dataset via Add Input"
DATA_DIR = os.path.dirname(hits[0])
print("Dataset folder:", DATA_DIR)

label_list = ["constant", "logn", "linear", "nlogn", "quadratic", "cubic"]   # order = complexity ranking
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
print("device:", device)

Dataset folder: /kaggle/input/datasets/sandrahanysamir/claude-data
device: cuda


## Cell 3: Structural labels from the code, and a family-held-out validation split
The parser extracts 5 properties from each Java method: maximum loop depth (0-3), an inner loop whose bound depends on an outer loop variable, a halving/doubling update (including binary-search style `lo = mid + 1`), recursion, and a `sort` call. They are rule-based and not hand-verified, so treat them as structural hints.

Each method also gets a one-line text summary (`struct_text`) that E4 gives to the model as input.

In [3]:
import javalang
from javalang import tree as T

LOOPS = (T.ForStatement, T.WhileStatement, T.DoStatement)

def _children(n):
    for c in n.children:
        if isinstance(c, T.Node):
            yield c
        elif isinstance(c, (list, tuple, set)):
            for x in c:
                if isinstance(x, T.Node):
                    yield x
                elif isinstance(x, (list, tuple)):
                    for y in x:
                        if isinstance(y, T.Node):
                            yield y

def _names(n):
    out = set()
    stack = [n]
    while stack:
        x = stack.pop()
        if isinstance(x, T.MemberReference):
            out.add(x.member)
        if isinstance(x, T.Node):
            stack.extend(_children(x))
    return out

def _has_div2(expr):
    """expression contains x / 2 or x >> 1 (midpoint / halving)"""
    stack = [expr]
    while stack:
        x = stack.pop()
        if isinstance(x, T.BinaryOperation):
            if x.operator == "/" and isinstance(x.operandr, T.Literal) and x.operandr.value in ("2", "2L"):
                return True
            if x.operator in (">>", ">>>") and isinstance(x.operandr, T.Literal) and x.operandr.value in ("1", "1L"):
                return True
        if isinstance(x, T.Node):
            stack.extend(_children(x))
    return False

def _control(loop):
    """the part of a loop that decides how many times it runs"""
    if isinstance(loop, T.ForStatement):
        return loop.control
    return loop.condition

def _declared(loop):
    if isinstance(loop, T.ForStatement):
        c = loop.control
        if isinstance(c, T.ForControl) and isinstance(c.init, T.VariableDeclaration):
            return {d.name for d in c.init.declarators}
        if isinstance(c, T.EnhancedForControl):
            return {d.name for d in c.var.declarators}
    return set()

def extract(src):
    """returns (depth_class 0..3, dependent, halving, recursion, sort) or None if the code can't be parsed"""
    try:
        tree = javalang.parse.parse("public class W {\n" + src + "\n}")
        methods = [m for _, m in tree.filter(T.MethodDeclaration)]
        if not methods:
            return None
        m = methods[0]
    except Exception:
        return None
    name = m.name
    st = {"depth": 0, "dep": 0, "half": 0, "rec": 0, "sort": 0}

    def walk(n, depth, outer_vars, mids):
        if isinstance(n, LOOPS):
            depth += 1
            st["depth"] = max(st["depth"], depth)
            if outer_vars and (_names(_control(n)) & outer_vars):
                st["dep"] = 1
            outer_vars = outer_vars | _declared(n)
        if isinstance(n, T.MethodInvocation):
            if n.member == name and not n.qualifier:
                st["rec"] = 1
            if n.member == "sort":
                st["sort"] = 1
        if isinstance(n, T.Assignment):
            ex = n.value
            if n.type in ("/=", "*=") and isinstance(ex, T.Literal) and ex.value not in ("1", "1L"):
                st["half"] = 1
            if n.type in ("<<=", ">>=", ">>>=") and isinstance(ex, T.Literal):
                st["half"] = 1
            if n.type == "=" and isinstance(n.expressionl, T.MemberReference):
                lhs = n.expressionl.member
                rhs = _names(ex)
                if isinstance(ex, T.BinaryOperation) and lhs in rhs and ex.operator in ("/", "*", ">>", "<<", ">>>"):
                    for side in (ex.operandl, ex.operandr):
                        if isinstance(side, T.Literal) and side.value not in ("0", "1", "1L"):
                            st["half"] = 1
                if (rhs & mids) and lhs not in mids:      # binary-search style: lo = mid + 1, hi = mid - 1
                    st["half"] = 1
        for c in _children(n):
            walk(c, depth, outer_vars, mids)

    mids = set()
    for _, d in m.filter(T.VariableDeclarator):
        if d.initializer is not None and _has_div2(d.initializer):
            mids.add(d.name)
    for _, a in m.filter(T.Assignment):
        if a.type == "=" and isinstance(a.expressionl, T.MemberReference) and _has_div2(a.value):
            mids.add(a.expressionl.member)

    walk(m, 0, set(), mids)
    return (min(st["depth"], 3), st["dep"], st["half"], st["rec"], st["sort"])


# ---------------- load ----------------
def load(name):
    return pd.read_json(os.path.join(DATA_DIR, name), lines=True)

train_raw, val_raw = load("train.jsonl"), load("validation.jsonl")
test_df, ext_df = load("test.jsonl"), load("external_test.jsonl")
for d in (train_raw, val_raw, test_df, ext_df):
    assert set(d["complexity"]) <= set(label_list)

pool = pd.concat([train_raw, val_raw], ignore_index=True)
gss = GroupShuffleSplit(n_splits=1, test_size=VAL_FAMILY_FRACTION, random_state=SPLIT_SEED)
fit_idx, val_idx = next(gss.split(pool, groups=pool["algorithm_family"]))
fit_df, val_df = pool.iloc[fit_idx].reset_index(drop=True), pool.iloc[val_idx].reset_index(drop=True)
assert not (set(fit_df["algorithm_family"]) & set(val_df["algorithm_family"]))
print("families  fit:", fit_df["algorithm_family"].nunique(), "| val:", val_df["algorithm_family"].nunique())

def struct_text(f):
    if f is None:
        return "[structure] unknown"
    return f"[structure] loop_depth={f[0]} dependent_loop={f[1]} halving_or_doubling={f[2]} recursion={f[3]} sort_call={f[4]}"

def add_aux(d):
    feats = [extract(s) for s in d["src"]]
    d = d.copy()
    d["aux_depth"] = [f[0] if f else -100 for f in feats]
    d["aux_bin"]   = [list(map(float, f[1:])) if f else [-1.0] * 4 for f in feats]
    d["struct_text"] = [struct_text(f) for f in feats]
    return d, sum(f is None for f in feats)

fit_df, a = add_aux(fit_df); val_df, b = add_aux(val_df)
test_df, c = add_aux(test_df); ext_df, e = add_aux(ext_df)
print("parse failures (fit/val/test/ext):", a, b, c, e)
for name, d in [("fit", fit_df), ("val", val_df), ("test", test_df), ("external_test", ext_df)]:
    print(f"{name:14s} {len(d):5d}", d["complexity"].value_counts().to_dict())
print("\nExample structure text:", fit_df["struct_text"].iloc[0])

families  fit: 59 | val: 11
parse failures (fit/val/test/ext): 0 0 0 0
fit             4994 {'linear': 934, 'nlogn': 922, 'quadratic': 833, 'logn': 802, 'cubic': 780, 'constant': 723}
val              958 {'constant': 269, 'cubic': 212, 'logn': 190, 'quadratic': 159, 'nlogn': 70, 'linear': 58}
test            1050 {'cubic': 175, 'quadratic': 175, 'linear': 175, 'constant': 175, 'logn': 175, 'nlogn': 175}
external_test    144 {'logn': 24, 'cubic': 24, 'nlogn': 24, 'linear': 24, 'constant': 24, 'quadratic': 24}

Example structure text: [structure] loop_depth=1 dependent_loop=0 halving_or_doubling=1 recursion=0 sort_call=0


## Cell 4: Tokenize (one version with the structure text, one without)

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
PAD = tokenizer.pad_token_id

class JavaDS(Dataset):
    def __init__(self, d, use_struct):
        texts = (d["struct_text"] + "\n" + d["src"]).tolist() if use_struct else d["src"].tolist()
        self.ids = tokenizer(texts, truncation=True, max_length=MAXLEN)["input_ids"]
        self.y = d["complexity"].map(label2id).values if "complexity" in d else np.zeros(len(d), dtype=int)
        self.depth = d["aux_depth"].values if "aux_depth" in d else np.full(len(d), -100)
        self.bins = np.array(d["aux_bin"].tolist(), dtype=np.float32) if "aux_bin" in d else np.full((len(d), 4), -1, dtype=np.float32)
    def __len__(self): return len(self.ids)
    def __getitem__(self, i): return self.ids[i], self.y[i], self.depth[i], self.bins[i]

def collate(batch):
    L = max(len(b[0]) for b in batch)
    ids = torch.full((len(batch), L), PAD, dtype=torch.long)
    mask = torch.zeros((len(batch), L), dtype=torch.long)
    for i, b in enumerate(batch):
        ids[i, :len(b[0])] = torch.tensor(b[0]); mask[i, :len(b[0])] = 1
    y = torch.tensor([b[1] for b in batch], dtype=torch.long)
    depth = torch.tensor([b[2] for b in batch], dtype=torch.long)
    bins = torch.tensor(np.stack([b[3] for b in batch]), dtype=torch.float32)
    return ids, mask, y, depth, bins

sets = {"fit": fit_df, "val": val_df, "test": test_df, "external_test": ext_df}
DS = {us: {k: JavaDS(v, us) for k, v in sets.items()} for us in (False, True)}
lens = [len(x) for x in DS[True]["fit"].ids]
print("tokenized. With structure text: max length", max(lens), "| truncated:", sum(l >= MAXLEN for l in lens))

config.json:   0%|          | 0.00/691 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.11k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/938k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/444k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

tokenized. With structure text: max length 421 | truncated: 0


## Cell 5: Model and training loop
One shared UniXcoder encoder with three heads: Big-O class (6), loop depth (4), 4 yes/no structural flags. The side-task heads are only trained when `lam > 0` (E3).

In [5]:
class MultiTaskModel(nn.Module):
    def __init__(self, name, n_cls=6, n_depth=4, n_bin=4):
        super().__init__()
        self.enc = AutoModel.from_pretrained(name)
        h = self.enc.config.hidden_size
        self.cls_head = nn.Sequential(nn.Dropout(0.1), nn.Linear(h, h), nn.Tanh(), nn.Dropout(0.1), nn.Linear(h, n_cls))
        self.depth_head = nn.Linear(h, n_depth)
        self.bin_head = nn.Linear(h, n_bin)
    def forward(self, input_ids, attention_mask):
        x = self.enc(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0]
        return self.cls_head(x), self.depth_head(x), self.bin_head(x)

def compute_loss(out, y, depth, bins, lam):
    cls, dl, bl = out
    main = F.cross_entropy(cls.float(), y)
    if lam == 0:
        return main, main.detach(), torch.zeros((), device=cls.device)
    m = depth != -100
    aux_d = F.cross_entropy(dl.float()[m], depth[m]) if m.any() else torch.zeros((), device=cls.device)
    mb = (bins >= 0).float()
    aux_b = (F.binary_cross_entropy_with_logits(bl.float(), bins.clamp(min=0), reduction="none") * mb).sum() / mb.sum().clamp(min=1)
    aux = aux_d + aux_b
    return main + lam * aux, main.detach(), aux.detach()

@torch.no_grad()
def predict_probs(model, ds, bs=32):
    model.eval()
    dl = DataLoader(ds, batch_size=bs, shuffle=False, collate_fn=collate)
    P = []
    for ids, mask, *_ in dl:
        with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
            cls, _, _ = model(ids.to(device), mask.to(device))
        P.append(torch.softmax(cls.float(), -1).cpu().numpy())
    return np.concatenate(P)

def predict_codes(model, codes, use_struct):
    d = pd.DataFrame({"src": list(codes)})
    d["struct_text"] = [struct_text(extract(s)) for s in d["src"]]
    return predict_probs(model, JavaDS(d, use_struct))

def train_one(lam, use_struct, seed, tag):
    set_seed(seed)
    ds_fit, ds_val = DS[use_struct]["fit"], DS[use_struct]["val"]
    model = MultiTaskModel(MODEL_NAME).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    dl = DataLoader(ds_fit, batch_size=BATCH, shuffle=True, collate_fn=collate,
                    generator=torch.Generator().manual_seed(seed))
    total = len(dl) * EPOCHS
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * total), total)
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    best_f1, best_state, bad = -1, None, 0
    for ep in range(1, EPOCHS + 1):
        model.train(); t0 = time.time(); tl = 0
        for ids, mask, y, depth, bins in dl:
            ids, mask, y, depth, bins = ids.to(device), mask.to(device), y.to(device), depth.to(device), bins.to(device)
            with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
                out = model(ids, mask)
            loss, main, aux = compute_loss(out, y, depth, bins, lam)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            tl += main.item()
        pred = predict_probs(model, ds_val).argmax(1)
        f1 = f1_score(ds_val.y, pred, average="macro"); acc = accuracy_score(ds_val.y, pred)
        print(f"  [{tag}] epoch {ep}/{EPOCHS} | train loss {tl/len(dl):.3f} | val(held-out families) acc {acc:.3f} F1 {f1:.3f} | {time.time()-t0:.0f}s")
        if f1 > best_f1:
            best_f1, bad = f1, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    model.load_state_dict(best_state)
    return model, best_f1

# ---- evaluation helpers ----
def p_worse(p_old, p_new):
    below = np.concatenate([np.zeros((len(p_old), 1)), np.cumsum(p_old, axis=1)[:, :-1]], axis=1)
    return (p_new * below).sum(axis=1)

def flag_scores(P, ranks, n_pairs=2000, seed=0):
    rng = np.random.RandomState(seed)
    a, b = rng.randint(0, len(P), n_pairs), rng.randint(0, len(P), n_pairs)
    k = a != b; a, b = a[k], b[k]
    truth = ranks[b] > ranks[a]
    pw = p_worse(P[a], P[b])
    def prf(flag):
        tp = (flag & truth).sum(); fp = (flag & ~truth).sum(); fn = (~flag & truth).sum()
        pr = tp / max(tp + fp, 1); rc = tp / max(tp + fn, 1)
        return pr, rc, 2 * pr * rc / max(pr + rc, 1e-9)
    return prf(pw > 0.5), prf(pw > 0.8)

def evaluate(model, use_struct, exp, seed):
    out = []
    for sname in ("test", "external_test"):
        ds = DS[use_struct][sname]
        P = predict_probs(model, ds)
        y, yp = ds.y, P.argmax(1)
        f05, f08 = flag_scores(P, y)
        cm = confusion_matrix(y, yp, labels=range(6))
        row = {"exp": exp, "seed": seed, "set": sname,
               "accuracy": accuracy_score(y, yp), "macro_F1": f1_score(y, yp, average="macro"),
               "within_1_class": float(np.mean(np.abs(y - yp) <= 1)),
               "flag_F1@0.5": f05[2], "flag_prec@0.8": f08[0], "flag_rec@0.8": f08[1]}
        for i, l in enumerate(label_list):
            row["recall_" + l] = cm[i, i] / max(cm[i].sum(), 1)
        out.append(row)
    return out

## Cell 6: Run all experiments and seeds
Prints one line per epoch. After each run the model is evaluated on `test` and `external_test`, and its results are stored. The model from the first seed of each experiment is kept for Cells 9-10; the others are freed to save GPU memory.

In [6]:
if "run_rows" not in globals():
    run_rows, kept, done = [], {}, set()
rows = run_rows

for seed in SEEDS:
    for exp, (lam, use_struct) in EXPERIMENTS.items():
        if (exp, seed) in done:
            continue
        print(f"=== {exp} | seed {seed} ===")
        model, bf1 = train_one(lam, use_struct, seed, f"{exp[:2]} s{seed}")
        rows += evaluate(model, use_struct, exp, seed)
        r = [x for x in rows if x["exp"] == exp and x["seed"] == seed and x["set"] == "external_test"][0]
        print(f"  best val F1 {bf1:.3f} | external_test acc {r['accuracy']:.3f} F1 {r['macro_F1']:.3f}")
        if seed == SEEDS[0]:
            kept[exp] = model
        else:
            del model
        gc.collect(); torch.cuda.empty_cache()
        done.add((exp, seed))
print("all runs finished")

=== E2 code only | seed 42 ===


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  504MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  504MB            

model.safetensors: downloading bytes:           |  0.00B            

... (14 outputs left out)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  [E3 s44] epoch 1/5 | train loss 0.648 | val(held-out families) acc 0.670 F1 0.560 | 74s
  [E3 s44] epoch 2/5 | train loss 0.006 | val(held-out families) acc 0.677 F1 0.596 | 74s
  [E3 s44] epoch 3/5 | train loss 0.001 | val(held-out families) acc 0.754 F1 0.640 | 74s
  [E3 s44] epoch 4/5 | train loss 0.001 | val(held-out families) acc 0.717 F1 0.612 | 75s
  [E3 s44] epoch 5/5 | train loss 0.000 | val(held-out families) acc 0.714 F1 0.610 | 74s
  best val F1 0.640 | external_test acc 0.632 F1 0.596
=== E4 code+structure | seed 44 ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  [E4 s44] epoch 1/5 | train loss 0.615 | val(held-out families) acc 0.732 F1 0.620 | 81s
  [E4 s44] epoch 2/5 | train loss 0.003 | val(held-out families) acc 0.724 F1 0.613 | 81s
  [E4 s44] epoch 3/5 | train loss 0.001 | val(held-out families) acc 0.741 F1 0.622 | 81s
  [E4 s44] epoch 4/5 | train loss 0.000 | val(held-out families) acc 0.751 F1 0.629 | 82s
  [E4 s44] epoch 5/5 | train loss 0.000 | val(held-out families) acc 0.751 F1 0.629 | 81s
  best val F1 0.629 | external_test acc 0.674 F1 0.657
all runs finished


## Cell 7: Summary over seeds (mean +- std)
`external_test` (unseen algorithm families) is the main number.

In [7]:
res = pd.DataFrame(rows)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)
metrics = ["accuracy", "macro_F1", "within_1_class", "flag_F1@0.5", "flag_prec@0.8", "flag_rec@0.8"]

def summarize(df):
    g = df.groupby("exp")[metrics].agg(["mean", "std"])
    out = pd.DataFrame(index=g.index)
    for m in metrics:
        out[m] = [f"{a:.3f} +- {b:.3f}" for a, b in zip(g[m]["mean"], g[m]["std"].fillna(0))]
    return out

for sname in ("external_test", "test"):
    print("=" * 30, sname, "=" * 30)
    print(summarize(res[res["set"] == sname]).to_string())

print("\nPer-seed external_test accuracy:")
print(res[res["set"] == "external_test"].pivot(index="seed", columns="exp", values="accuracy").round(3).to_string())

print("\nMean recall per class on external_test:")
rc = ["recall_" + l for l in label_list]
print(res[res["set"] == "external_test"].groupby("exp")[rc].mean().round(2).rename(columns=lambda c: c[7:]).to_string())

============================== external_test ==============================
                         accuracy        macro_F1  within_1_class     flag_F1@0.5   flag_prec@0.8    flag_rec@0.8
exp                                                                                                              
E2 code only       0.694 +- 0.032  0.678 +- 0.036  0.863 +- 0.017  0.802 +- 0.007  0.831 +- 0.006  0.754 +- 0.027
E3 side task       0.678 +- 0.041  0.641 +- 0.041  0.917 +- 0.018  0.837 +- 0.021  0.860 +- 0.004  0.773 +- 0.014
E4 code+structure  0.697 +- 0.024  0.675 +- 0.029  0.898 +- 0.011  0.836 +- 0.012  0.861 +- 0.015  0.782 +- 0.023
============================== test ==============================
                         accuracy        macro_F1  within_1_class     flag_F1@0.5   flag_prec@0.8    flag_rec@0.8
exp                                                                                                              
E2 code only       0.930 +- 0.017  0.929 +- 0.017  0.983 +-

## Cell 8: Is E4 really better than E2?
Paired by seed on `external_test` (same seed = same data order). With only 3 seeds and 144 test samples this is a rough guide, not a proof.

In [8]:
ext = res[res["set"] == "external_test"].pivot(index="seed", columns="exp", values="accuracy")
for other in ["E3 side task", "E4 code+structure"]:
    d = (ext[other] - ext["E2 code only"])
    print(f"{other} minus E2 (accuracy, per seed):", np.round(d.values, 3),
          "| mean", round(float(d.mean()), 3), "| better in", int((d > 0).sum()), "of", len(d), "seeds")
d = ext["E4 code+structure"] - ext["E3 side task"]
print("E4 minus E3:", np.round(d.values, 3), "| mean", round(float(d.mean()), 3))
print("\nRule of thumb: external_test has 144 samples, so 1 sample = 0.007 accuracy. "
      "Differences of a few samples that do not repeat across seeds are noise.")

E3 side task minus E2 (accuracy, per seed): [ 0.007 -0.028 -0.028] | mean -0.016 | better in 1 of 3 seeds
E4 code+structure minus E2 (accuracy, per seed): [ 0.021 -0.028  0.014] | mean 0.002 | better in 2 of 3 seeds
E4 minus E3: [0.014 0.    0.042] | mean 0.019

Rule of thumb: external_test has 144 samples, so 1 sample = 0.007 accuracy. Differences of a few samples that do not repeat across seeds are noise.


## Cell 9: Save the E4 model (code + structure) and check the save works

In [9]:
m4 = kept["E4 code+structure"]
os.makedirs(SAVE_DIR, exist_ok=True)
torch.save(m4.state_dict(), os.path.join(SAVE_DIR, "model.pt"))
tokenizer.save_pretrained(SAVE_DIR)
json.dump({"model_name": MODEL_NAME, "label_list": label_list, "input": "struct_text + newline + code"},
          open(os.path.join(SAVE_DIR, "meta.json"), "w"))
print(os.listdir(SAVE_DIR))

reloaded = MultiTaskModel(MODEL_NAME).to(device)
reloaded.load_state_dict(torch.load(os.path.join(SAVE_DIR, "model.pt"), map_location=device))
P1 = predict_probs(m4, DS[True]["external_test"]); P2 = predict_probs(reloaded, DS[True]["external_test"])
print("reload check, max difference in probabilities:", float(np.abs(P1 - P2).max()))

['meta.json', 'tokenizer.json', 'model.pt', 'tokenizer_config.json']


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

reload check, max difference in probabilities: 0.0


## Cell 10: Try a before/after pair with all three models

In [10]:
old_code = """
public static int find(int[] a, int t) {
    for (int i = 0; i < a.length; i++)
        if (a[i] == t) return i;
    return -1;
}
"""
new_code = """
public static int find(int[] a, int t) {
    int c = 0;
    for (int i = 0; i < a.length; i++)
        for (int j = 0; j < a.length; j++)
            if (a[i] + a[j] == t) c++;
    return c;
}
"""
print("structure old:", struct_text(extract(old_code)))
print("structure new:", struct_text(extract(new_code)), "\n")
for exp, (lam, use_struct) in EXPERIMENTS.items():
    po, pn = predict_codes(kept[exp], [old_code], use_struct), predict_codes(kept[exp], [new_code], use_struct)
    print(f"{exp:20s} old -> {id2label[int(po.argmax())]:9s} new -> {id2label[int(pn.argmax())]:9s} P(new slower) = {float(p_worse(po, pn)[0]):.3f}")

structure old: [structure] loop_depth=1 dependent_loop=0 halving_or_doubling=0 recursion=0 sort_call=0
structure new: [structure] loop_depth=2 dependent_loop=0 halving_or_doubling=0 recursion=0 sort_call=0 

E2 code only         old -> linear    new -> cubic     P(new slower) = 1.000
E3 side task         old -> linear    new -> cubic     P(new slower) = 0.997
E4 code+structure    old -> linear    new -> cubic     P(new slower) = 1.000
